### Week 07 Excercise Solution End to End

## Install Required Packages and Download Helper Utilities

This cell prepares the notebook environment by installing the required libraries and downloading a helper utility file.

### Package Installation

The following packages are installed:

- **bitsandbytes==0.48.2**  
  Used for efficient model loading, quantization, and memory optimization when working with large language models.

- **trl==0.25.1**  
  A library for transformer-based reinforcement learning and supervised fine-tuning workflows.

The `-q` flag keeps the installation output quiet, and `--upgrade` ensures the specified versions are installed.

### Downloading the Utility File

The notebook also downloads a helper script from the course repository:

- **util.py**

This file is fetched from the `llm_engineering` GitHub repository and saved locally so it can be imported in later cells.

### Why This Step Matters

This setup step ensures that:
- the correct package versions are available
- helper functions from the course repository can be reused
- the notebook environment is ready for the Week 07 workflow

In [ ]:
# Install required libraries for Week 07
# - bitsandbytes: used for efficient model loading and quantization
# - trl: used for transformer reinforcement learning / supervised fine-tuning workflows
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

# Download the helper utility file from the course GitHub repository
# The file is saved locally as util.py
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

## Import Libraries for Model Training and Fine-Tuning

This cell imports the main libraries required for the Week 07 workflow, which focuses on training and fine-tuning large language models.

### 1. Standard Python Libraries

The notebook first imports basic Python utilities:

- **os** — file system and environment variable management  
- **re** — regular expression processing for text cleaning  
- **math** — mathematical functions  

These libraries support general data processing tasks.

### 2. Progress Monitoring

`tqdm` provides progress bars for loops and long-running operations, which helps monitor training or preprocessing tasks.

### 3. Authentication and Credentials

Two utilities are used for authentication:

- **google.colab.userdata** — access secure credentials stored in Colab  
- **huggingface_hub.login** — authenticate with Hugging Face to access models or datasets

### 4. PyTorch

PyTorch is the deep learning framework used for training neural networks and transformer models.

It provides:
- tensor operations
- GPU acceleration
- neural network training tools

### 5. Hugging Face Transformers

The **transformers** library provides pre-trained language models and training utilities.

Key components imported include:

- **AutoModelForCausalLM** — loads causal language models such as LLaMA or GPT-style models  
- **AutoTokenizer** — converts text into token IDs that models can process  
- **TrainingArguments** — defines training configuration such as batch size, learning rate, and logging  
- **set_seed** — ensures reproducibility  
- **BitsAndBytesConfig** — enables quantized model loading to reduce memory usage

### 6. Dataset Handling

The **datasets** library allows easy loading and manipulation of datasets.

Important classes include:

- **load_dataset** — loads datasets from Hugging Face Hub  
- **Dataset** — represents a single dataset split  
- **DatasetDict** — manages multiple splits such as train/validation/test

### 7. Experiment Tracking

**Weights & Biases (wandb)** is used for experiment tracking, including:

- logging training metrics
- visualizing learning curves
- comparing experiments

### 8. Parameter-Efficient Fine-Tuning (PEFT)

The **peft** library provides tools for efficient fine-tuning of large models.

`LoraConfig` is used to configure **LoRA (Low-Rank Adaptation)**, which reduces the number of trainable parameters during fine-tuning.

### 9. TRL – Supervised Fine-Tuning

The **trl** library provides tools for training language models.

Key components:

- **SFTTrainer** — trainer class for supervised fine-tuning  
- **SFTConfig** — configuration settings for SFT training

### 10. Visualization and Experiment Metadata

Additional utilities include:

- **datetime** — useful for timestamping experiments  
- **matplotlib** — plotting training curves and evaluation metrics

### Summary

This cell prepares the environment for the Week 07 fine-tuning workflow by importing tools for:

- dataset loading
- model loading
- parameter-efficient training
- experiment tracking
- result visualization

In [ ]:
# Import standard libraries used for file operations, regex processing, and math utilities
import os
import re
import math

# Progress bar utility for loops
from tqdm import tqdm

# Access secrets stored in Google Colab (API keys etc.)
from google.colab import userdata

# Hugging Face login utility for accessing models and datasets
from huggingface_hub import login

# PyTorch library for deep learning
import torch

# Hugging Face Transformers library
import transformers

# Import key transformer components
from transformers import (
    AutoModelForCausalLM,   # Load causal language models
    AutoTokenizer,          # Load tokenizers for text processing
    TrainingArguments,      # Configure training settings
    set_seed,               # Set reproducibility seed
    BitsAndBytesConfig      # Configure model quantization
)

# Dataset utilities from Hugging Face
from datasets import load_dataset, Dataset, DatasetDict

# Weights & Biases for experiment tracking
import wandb

# PEFT library for parameter-efficient fine-tuning (LoRA)
from peft import LoraConfig

# TRL (Transformer Reinforcement Learning) tools for supervised fine-tuning
from trl import SFTTrainer, SFTConfig

# Utility for timestamping experiments
from datetime import datetime

# Plotting library for visualizing training results
import matplotlib.pyplot as plt

## Training Configuration and Hyperparameters

This cell defines the configuration parameters used for the Week 07 fine-tuning experiment.

It includes settings for:

- base model selection
- dataset configuration
- experiment tracking
- LoRA fine-tuning
- training hyperparameters

---

## 1. Base Model

The model being fine-tuned is:


This is a **3B parameter LLaMA model** used as the starting point for training.

Instead of training a model from scratch, the notebook performs **parameter-efficient fine-tuning** using LoRA.

---

## 2. Project and Run Naming

Several variables are used to track experiments:

- **PROJECT_NAME** → identifies the project
- **RUN_NAME** → timestamped run identifier
- **PROJECT_RUN_NAME** → combined project + run name
- **HUB_MODEL_NAME** → final Hugging Face model repository name

Example:


This makes experiment tracking easier when multiple runs are performed.

---

## 3. Lite Mode

`LITE_MODE = True` enables a smaller experiment setup.

This typically means:

- smaller datasets
- faster training
- reduced compute requirements

This is useful when testing configurations before running full-scale training.

---

## 4. Dataset Configuration

The dataset used in this experiment is:


This dataset contains product prompts used for training the pricing model.

---

## 5. General Training Hyperparameters

Important training settings include:

- **EPOCHS** → number of passes through the dataset
- **BATCH_SIZE** → number of samples per training batch
- **MAX_SEQUENCE_LENGTH** → maximum token length for input prompts
- **GRADIENT_ACCUMULATION_STEPS** → allows larger effective batch sizes

---

## 6. QLoRA Configuration

The model uses **QLoRA**, which combines:

- **4-bit quantization**
- **LoRA adapters**

This allows large models to be fine-tuned using much less GPU memory.

Key parameters include:

- **LORA_R** → rank of LoRA adapters
- **LORA_ALPHA** → scaling factor
- **TARGET_MODULES** → transformer layers where LoRA adapters are inserted
- **LORA_DROPOUT** → regularization to reduce overfitting

In this configuration, LoRA adapters are applied to **attention projection layers**.

---

## 7. Training Optimizer Settings

The training process uses:

- **learning rate:** `1e-4`
- **cosine learning rate scheduler**
- **weight decay:** `0.001`
- **optimizer:** `paged_adamw_32bit`

The paged optimizer helps reduce memory usage when training large models.

---

## 8. GPU Capability Detection

The code checks GPU capability to determine if **bfloat16 (bf16)** precision can be used.


bf16 training improves performance on modern GPUs such as:

- A100
- H100
- RTX 40 series

---

## 9. Experiment Tracking

The following parameters control experiment monitoring:

- **VAL_SIZE** → number of validation samples
- **LOG_STEPS** → logging frequency
- **SAVE_STEPS** → checkpoint saving frequency
- **LOG_TO_WANDB** → enables Weights & Biases experiment tracking

This helps track training progress and compare multiple runs.

---

## Summary

This cell defines the **full configuration for the fine-tuning experiment**, including:

- model selection
- LoRA training parameters
- dataset configuration
- optimizer settings
- experiment tracking

These settings control how the LLaMA model will be fine-tuned for the pricing task.

In [ ]:
# Constants and configuration parameters for the training run

# Base model that will be fine-tuned
BASE_MODEL = "meta-llama/Llama-3.2-3B"

# Project name used for experiment tracking
PROJECT_NAME = "price"

# Hugging Face username (replace with your own HF account name)
HF_USER = "olasogba"

# Enable lite mode for faster experiments with smaller datasets
LITE_MODE = True


# Dataset configuration
DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite"


# Generate a timestamped run name for experiment tracking
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"

# Append "-lite" suffix if lite mode is enabled
if LITE_MODE:
    RUN_NAME += "-lite"

# Combine project name and run name
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"

# Hugging Face Hub model name for uploading the trained model
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# ------------------------------
# General training hyperparameters
# ------------------------------

EPOCHS = 1                      # number of training epochs
BATCH_SIZE = 32                 # batch size for training
MAX_SEQUENCE_LENGTH = 128       # maximum token sequence length
GRADIENT_ACCUMULATION_STEPS = 1 # accumulate gradients over batches


# ------------------------------
# QLoRA configuration parameters
# ------------------------------

QUANT_4_BIT = True              # enable 4-bit quantization

# LoRA rank (controls adapter size)
LORA_R = 32

# LoRA scaling factor
LORA_ALPHA = LORA_R * 2

# Transformer attention layers to apply LoRA adapters
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]

# MLP layers (not used here but listed for optional extension)
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]

# Target modules for LoRA training
TARGET_MODULES = ATTENTION_LAYERS

# Dropout applied to LoRA layers
LORA_DROPOUT = 0.1


# ------------------------------
# Training optimizer parameters
# ------------------------------

LEARNING_RATE = 1e-4            # optimizer learning rate
WARMUP_RATIO = 0.01             # warmup proportion for scheduler
LR_SCHEDULER_TYPE = "cosine"    # learning rate scheduler type
WEIGHT_DECAY = 0.001            # regularization weight decay
OPTIMIZER = "paged_adamw_32bit" # memory-efficient optimizer


# Detect GPU capability to determine whether bfloat16 can be used
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8


# ------------------------------
# Experiment tracking settings
# ------------------------------

VAL_SIZE = 500      # number of validation examples
LOG_STEPS = 5       # how often to log training metrics
SAVE_STEPS = 100    # checkpoint saving frequency
LOG_TO_WANDB = True # enable Weights & Biases logging

## Authentication, Dataset Loading, and Quantization Setup

This section performs three important setup tasks:

1. authentication with external services  
2. loading the training dataset  
3. configuring model quantization for efficient fine-tuning

---

## 1. Hugging Face Authentication

The notebook retrieves the **Hugging Face API token** from Colab secrets:


This token allows the notebook to:

- download models and datasets
- push trained models back to Hugging Face Hub

The command


authenticates the session with the Hugging Face Hub.

---

## 2. Weights & Biases Authentication

Weights & Biases (wandb) is used for experiment tracking.

The notebook retrieves the API key from Colab secrets:


The key is stored as an environment variable and then used to log in.

Additional configuration variables control how wandb behaves:

- **WANDB_PROJECT** → name of the tracking project
- **WANDB_LOG_MODEL** → disables automatic model uploads
- **WANDB_WATCH** → disables gradient monitoring

If experiment tracking is enabled (`LOG_TO_WANDB = True`), the notebook starts a new run:


Each run is tagged with the timestamped run name defined earlier.

---

## 3. Loading the Dataset

The dataset is loaded from Hugging Face using:


examples.

---

## 4. 4-bit Quantization Configuration (QLoRA)

To make training large models feasible on smaller GPUs, the notebook uses **QLoRA**, which combines:

- **LoRA adapters**
- **4-bit quantization**

The configuration is defined using `BitsAndBytesConfig`.

Key settings:

- **load_in_4bit=True**  
  loads the model using 4-bit precision

- **bnb_4bit_use_double_quant=True**  
  applies nested quantization for improved compression

- **bnb_4bit_compute_dtype**  
  determines whether computation uses:
  - `bfloat16` (if supported)
  - `float16` otherwise

- **bnb_4bit_quant_type="nf4"**  
  uses **Normal Float 4 (NF4)** quantization, which is optimized for large language models.

---

## Summary

This step prepares the training environment by:

- authenticating with Hugging Face and Weights & Biases
- loading the dataset used for fine-tuning
- configuring memory-efficient **4-bit quantization**

These steps allow the large LLaMA model to be fine-tuned efficiently even on limited GPU resources.

In [ ]:
# Log in to Hugging Face using the token stored in Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Authenticate with Hugging Face Hub
login(hf_token, add_to_git_credential=True)


# Log in to Weights & Biases for experiment tracking
wandb_api_key = userdata.get('WANDB_API_KEY')

# Set the API key as an environment variable
os.environ["WANDB_API_KEY"] = wandb_api_key

# Authenticate with the wandb service
wandb.login()


# Configure Weights & Biases project settings
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"   # prevent automatic model uploads
os.environ["WANDB_WATCH"] = "false"       # disable gradient watching


# Load the dataset from Hugging Face Hub
dataset = load_dataset(DATASET_NAME)

# Extract train, validation, and test splits
train = dataset['train']
val = dataset['val'].select(range(VAL_SIZE))  # limit validation size
test = dataset['test']


# Initialize wandb experiment run if tracking is enabled
if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)


# Configure 4-bit quantization for QLoRA training
if QUANT_4_BIT:

    quant_config = BitsAndBytesConfig(

        # Load the model using 4-bit quantization
        load_in_4bit=True,

        # Enable nested quantization for better memory efficiency
        bnb_4bit_use_double_quant=True,

        # Set computation dtype depending on GPU capability
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,

        # Use NF4 quantization (recommended for LLMs)
        bnb_4bit_quant_type="nf4"
    )

## Load the Model, Configure LoRA Training, and Fine-Tune

This section performs the main fine-tuning workflow:

1. load the tokenizer  
2. load the base model  
3. configure LoRA adapters  
4. define supervised fine-tuning parameters  
5. train the model  
6. push the trained model to Hugging Face Hub

---

## 1. Load the Tokenizer

The tokenizer is loaded from the selected base model:


Two important tokenizer settings are applied:

- **pad_token = eos_token**  
  This ensures that padding uses the end-of-sequence token.

- **padding_side = "right"**  
  Right-side padding is generally preferred for causal language model training.

These settings help avoid tokenization issues during batching and generation.

---

## 2. Load the Base Model

The base model is loaded using:


Important configuration choices:

- **quantization_config=quant_config**  
  applies the previously defined 4-bit quantization setup

- **device_map="auto"**  
  automatically distributes model layers across available devices

After loading, the model’s generation config is updated so the padding token matches the tokenizer.

The code also prints the model’s **memory footprint**, which is useful for checking whether the model fits within available GPU memory.

---

## 3. Configure LoRA Parameters

The `LoraConfig` object defines how parameter-efficient fine-tuning will be applied.

Key settings include:

- **lora_alpha** — scaling factor for LoRA updates  
- **lora_dropout** — dropout for regularization  
- **r** — rank of the LoRA adapters  
- **bias="none"** — no bias parameters are trained  
- **task_type="CAUSAL_LM"** — training is for causal language modeling  
- **target_modules** — transformer layers that receive LoRA adapters

This allows the model to be fine-tuned efficiently while updating only a small subset of parameters.

---

## 4. Configure SFT Training Parameters

The `SFTConfig` object defines the full supervised fine-tuning setup.

### Training settings
- number of epochs
- batch size
- gradient accumulation
- learning rate
- weight decay
- gradient clipping

### Precision settings
- **fp16** or **bf16** depending on hardware support

### Logging and checkpointing
- logging frequency
- evaluation frequency
- model checkpoint saving
- Weights & Biases reporting

### Hugging Face Hub integration
- automatically push checkpoints to the Hub
- save under a private repository name
- use the configured `HUB_MODEL_NAME`

This configuration controls the full fine-tuning workflow.

---

## 5. Create the Trainer

The fine-tuning process is managed by:


The trainer receives:

- the base model
- the training dataset
- the validation dataset
- the LoRA configuration
- the full training configuration

This trainer handles:
- batching
- evaluation
- checkpointing
- logging
- training loop execution

---

## 6. Fine-Tune the Model

The command


starts supervised fine-tuning of the quantized LLaMA model using LoRA adapters.

This is the main training step of the Week 07 notebook.

---

## 7. Push the Model to Hugging Face Hub

After training completes, the fine-tuned model is uploaded to a private Hugging Face Hub repository:


This makes it easier to:
- reuse the trained model later
- share results
- deploy or evaluate the model in future notebooks

---

## 8. Finish Weights & Biases Logging

If Weights & Biases tracking is enabled, the run is closed using:


This ensures that all metrics and logs are properly saved.

---

## Summary

This section executes the full parameter-efficient fine-tuning pipeline by:

- loading the tokenizer and model
- configuring LoRA adapters
- defining training hyperparameters
- running supervised fine-tuning
- uploading the final model to Hugging Face Hub
- closing experiment tracking

It is the core training stage of the Week 07 workflow.

In [ ]:
# Load the tokenizer for the base model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Set padding token to EOS token to avoid padding issues
tokenizer.pad_token = tokenizer.eos_token

# Use right-side padding for training
tokenizer.padding_side = "right"


# Load the base causal language model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,  # apply 4-bit quantization if enabled
    device_map="auto",                 # automatically place model layers on available devices
)

# Ensure generation uses the tokenizer pad token
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Print model memory footprint for visibility
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")


# ------------------------------
# Set up LoRA parameters
# ------------------------------

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,         # LoRA scaling factor
    lora_dropout=LORA_DROPOUT,     # Dropout for LoRA layers
    r=LORA_R,                      # LoRA rank
    bias="none",                   # Do not train bias terms
    task_type="CAUSAL_LM",         # Task type is causal language modeling
    target_modules=TARGET_MODULES, # Modules where LoRA adapters will be inserted
)


# ------------------------------
# Set up training configuration
# ------------------------------

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,                  # local output directory
    num_train_epochs=EPOCHS,                      # total number of epochs
    per_device_train_batch_size=BATCH_SIZE,       # training batch size per device
    per_device_eval_batch_size=1,                 # evaluation batch size
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,                              # optimizer type
    save_steps=SAVE_STEPS,                        # checkpoint save frequency
    save_total_limit=10,                          # max checkpoints to keep
    logging_steps=LOG_STEPS,                      # metric logging frequency
    learning_rate=LEARNING_RATE,                  # learning rate
    weight_decay=0.001,                           # L2 regularization
    fp16=not use_bf16,                            # use float16 if bf16 unavailable
    bf16=use_bf16,                                # use bfloat16 if supported
    max_grad_norm=0.3,                            # gradient clipping
    max_steps=-1,                                 # full epoch-based training
    warmup_ratio=WARMUP_RATIO,                    # LR warmup
    group_by_length=True,                         # batch examples of similar length
    lr_scheduler_type=LR_SCHEDULER_TYPE,          # scheduler type
    report_to="wandb" if LOG_TO_WANDB else None,  # report metrics to Weights & Biases
    run_name=RUN_NAME,                            # run name for experiment tracking
    max_length=MAX_SEQUENCE_LENGTH,               # max token length
    save_strategy="steps",                        # save model every fixed number of steps
    hub_strategy="every_save",                    # push checkpoints to HF Hub
    push_to_hub=True,                             # enable hub upload
    hub_model_id=HUB_MODEL_NAME,                  # repository name on HF Hub
    hub_private_repo=True,                        # keep repository private
    eval_strategy="steps",                        # evaluate every few steps
    eval_steps=SAVE_STEPS                         # evaluation frequency
)


# ------------------------------
# Create the supervised fine-tuning trainer
# ------------------------------

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters
)


# ------------------------------
# Start fine-tuning
# ------------------------------
fine_tuning.train()


# Push the final fine-tuned model to Hugging Face Hub
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)

print(f"Saved to the hub: {PROJECT_RUN_NAME}")


# Finish the Weights & Biases run if logging is enabled
if LOG_TO_WANDB:
    wandb.finish()

## Load and Evaluate the Fine-Tuned PEFT Model

This section loads the fine-tuned LoRA adapter, attaches it to the base model, and evaluates the resulting model on the test set.

---

## 1. Imports

Two utilities are imported:

- **PeftModel** — used to load the LoRA adapter on top of the base model
- **evaluate** — helper function from `util.py` used to test model predictions

These are the main tools needed for inference and evaluation.

---

## 2. Evaluation Configuration

The notebook defines a few evaluation settings:

- **RUN_NAME** — identifies the fine-tuned run to evaluate
- **REVISION** — optionally specifies a model version on Hugging Face Hub
- **QUANT_4_BIT** — keeps inference efficient using 4-bit quantization

GPU capability is checked again to determine whether **bf16** is supported.

---

## 3. Load the Test Dataset

The dataset is loaded again from Hugging Face Hub, and only the **test split** is used.

The first example is displayed so you can inspect the structure of each datapoint, especially the `prompt` field that will be passed to the model.

---

## 4. Load the Fine-Tuned PEFT Model

The fine-tuned LoRA adapter is loaded using:


This attaches the learned LoRA weights to the original base model.

The result is a full inference model that combines:

- the original quantized base model
- the task-specific fine-tuned LoRA adapter

The notebook then prints the model’s **memory footprint** and displays the full model structure.

---

## 5. Prediction Function

The function `model_predict(item)` defines how one dataset item is processed during evaluation.

### Steps performed:

1. tokenize the input prompt  
2. move the input tensors to GPU  
3. generate output tokens with the fine-tuned model  
4. remove the original prompt tokens from the generated output  
5. decode only the newly generated text

The function returns the model’s generated answer as plain text.

This wrapper is necessary because the evaluator expects a callable that accepts one item and returns a prediction.

---

## 6. Evaluation

The random seed is set for reproducibility:


This measures how well the fine-tuned model performs on the pricing task.

---

## Summary

This section completes the Week 07 workflow by:

- loading the trained LoRA adapter
- reconstructing the full fine-tuned model
- generating predictions from prompts
- evaluating performance on the test dataset

It is the final inference and validation step after supervised fine-tuning.

In [ ]:
# Import PEFT model loader and evaluation utility
from peft import PeftModel
from util import evaluate


# ------------------------------
# Constants for evaluation
# ------------------------------

# If lite mode is being used, define the run name of the trained adapter
if LITE_MODE:
    RUN_NAME = "2026-03-08_14.44.17-lite"

# Model revision on Hugging Face Hub (None means latest)
REVISION = None


# ------------------------------
# Quantization / hardware setup
# ------------------------------

# Keep 4-bit quantization enabled for efficient inference
QUANT_4_BIT = True

# Detect GPU capability to decide whether bf16 is supported
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8


# ------------------------------
# Load the evaluation dataset
# ------------------------------

dataset = load_dataset(DATASET_NAME)
test = dataset["test"]

# Display the first test sample to inspect the prompt format
test[0]


# ------------------------------
# Load the fine-tuned PEFT adapter
# ------------------------------

# Attach the fine-tuned LoRA adapter to the base model
fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)

# Print memory footprint of the loaded fine-tuned model
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

# Display model structure
fine_tuned_model


# ------------------------------
# Prediction function
# ------------------------------

def model_predict(item):
    # Tokenize the input prompt and move tensors to GPU
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")

    # Disable gradient computation during inference
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)

    # Remove the original prompt tokens from the generated sequence
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]

    # Decode only the generated output text
    return tokenizer.decode(generated_ids)


# ------------------------------
# Evaluate the fine-tuned model
# ------------------------------

# Set seed for reproducibility
set_seed(42)

# Run evaluation on the test set
evaluate(model_predict, test)